# Gaussian Splatting Backend

This notebook runs a **FastAPI server** on a Colab NVIDIA GPU that:

1. Receives photos and videos from the CaptureLab app
2. Runs **COLMAP** structure-from-motion ([colmap/colmap](https://github.com/colmap/colmap))
3. Trains **3D Gaussian Splatting** ([MrNeRF/LichtFeld-Studio](https://github.com/MrNeRF/LichtFeld-Studio))
4. Exports a web-viewable `.splat` file and `.ply` for the viewer
5. Serves a live 3D viewer URL back to the app ([playcanvas/supersplat](https://github.com/playcanvas/supersplat))

The server is exposed via a **Cloudflare Tunnel** (`api.gaussiansplat.dev`).

## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU - change Runtime type to any GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n GPU: {gpu_name} ({gpu_mem:.0f} GB)")

## 1.5 Mount Google Drive (persistent build cache)

First run: COLMAP and LichtFeld-Studio build from source (~1 hr) and the
binaries are uploaded to Drive.

Cache location: `Drive/MyDrive/gaussian-splat-env/cache/`. Delete that folder
to force a clean rebuild (e.g. after a Colab CUDA / driver upgrade).


In [ ]:
# Mount Google Drive once per session and expose the cache path to subsequent
# bash cells via the SPLAT_CACHE env var. Only the slow builds (COLMAP and
# LichtFeld-Studio) are cached
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
CACHE_ROOT = '/content/drive/MyDrive/gaussian-splat-env'
CACHE_DIR  = f'{CACHE_ROOT}/cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['SPLAT_CACHE'] = CACHE_DIR

print(f'Cache: {CACHE_DIR}')
for tag, path in [
    ('COLMAP (cuda)',     f'{CACHE_DIR}/colmap-cuda.tar.gz'),
    ('LichtFeld build',   f'{CACHE_DIR}/lichtfeld-build.tar.gz'),]:
    if os.path.exists(path):
        size = subprocess.check_output(['du', '-sh', path]).split()[0].decode()
        print(f'  [CACHED]  {tag:18s} {size}')
    else:
        print(f'  [empty]  {tag:18s} (will be built and cached on first run)')


## 2. Install System Dependencies

COLMAP built from source with CUDA support for structure-from-motion


In [ ]:
%%bash
set -e

CACHE_DIR="${SPLAT_CACHE:-/content/drive/MyDrive/gaussian-splat-env/cache}"
COLMAP_TAR="$CACHE_DIR/colmap-cuda.tar.gz"

# Runtime system deps 
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev \
    libeigen3-dev libflann-dev libfreeimage-dev \
    libmetis-dev libgoogle-glog-dev libgflags-dev \
    libsqlite3-dev libglew-dev libglvnd-dev \
    libcgal-dev libceres-dev \
    imagemagick > /dev/null 2>&1

# Try cache restore
if [ -f "$COLMAP_TAR" ]; then
    echo "Restoring COLMAP from Drive cache..."
    tar xzf "$COLMAP_TAR" -C /
    ldconfig 2>/dev/null || true
    if colmap feature_extractor --help 2>&1 | grep -q "use_gpu"; then
        echo "COLMAP restored: $(which colmap) (CUDA enabled)"
        exit 0
    fi
    echo "Cached COLMAP failed to run with CUDA -- rebuilding."
    rm -f "$COLMAP_TAR"
fi

# Build path
apt-get install -y -qq cmake ninja-build build-essential > /dev/null 2>&1

COLMAP_DIR="/content/colmap"
if [ ! -d "$COLMAP_DIR" ]; then
    git clone https://github.com/colmap/colmap.git "$COLMAP_DIR" --branch 3.11.1 --depth 1
fi

export PATH=/usr/local/cuda/bin:$PATH
export CUDACXX=/usr/local/cuda/bin/nvcc

cd "$COLMAP_DIR"
rm -rf build && mkdir build && cd build

echo "Configuring COLMAP with CUDA..."
cmake .. -GNinja \
    -DCMAKE_BUILD_TYPE=Release \
    -DGUI_ENABLED=OFF \
    -DCUDA_ENABLED=ON \
    -DCMAKE_EXE_LINKER_FLAGS="-lGLdispatch" \
    -DCMAKE_CUDA_ARCHITECTURES=native

echo "Building COLMAP..."
ninja -j$(nproc)
ninja install
ldconfig

echo "COLMAP installed: $(which colmap)"
if colmap feature_extractor --help 2>&1 | grep -q "use_gpu"; then
    echo "CUDA support: ENABLED"
else
    echo "ERROR: CUDA support is NOT enabled -- check cmake output above"
    exit 1
fi

# Save to Drive cache
echo "Caching COLMAP install to Drive..."
mkdir -p "$CACHE_DIR"
COLMAP_FILES=$({
    find /usr/local -path '*colmap*' 2>/dev/null
    find /usr/local -name 'libcolmap*' 2>/dev/null
} | sort -u)
if [ -n "$COLMAP_FILES" ]; then
    echo "$COLMAP_FILES" | tar czf "$COLMAP_TAR" -T - 2>/dev/null
    echo "  cached: $(du -h "$COLMAP_TAR" | cut -f1) at $COLMAP_TAR"
    echo "  files cached: $(echo "$COLMAP_FILES" | wc -l)"
else
    echo "  warning: could not locate installed COLMAP files; cache skipped"
fi

## 3. Build 3D Gaussian Splatting Engine

Uses [MrNeRF/LichtFeld-Studio](https://github.com/MrNeRF/LichtFeld-Studio) (C++/CUDA) for 3D Gaussian Splatting training.

In [ ]:
%%bash
set -e

CACHE_DIR="${SPLAT_CACHE:-/content/drive/MyDrive/gaussian-splat-env/cache}"
LF_TAR="$CACHE_DIR/lichtfeld-build.tar.gz"
LF_DIR="/content/LichtFeld-Studio"
VCPKG_ROOT="/content/vcpkg"

# Runtime system deps -- always needed
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq \
    libgl1-mesa-dev libglu1-mesa-dev libegl1-mesa-dev \
    libx11-dev libxft-dev libxext-dev libxrandr-dev libxinerama-dev libxcursor-dev libxi-dev libxtst-dev libxss-dev \
    libwayland-dev libxkbcommon-dev libdecor-0-dev libdrm-dev libgbm-dev \
    libibus-1.0-dev libdbus-1-dev libsystemd-dev libudev-dev \
    libasound2-dev libpulse-dev libpipewire-0.3-dev > /dev/null 2>&1

# Check if libstdc++ provides the required GLIBCXX/CXXABI versions, which are needed by some vcpkg dependencies
if ! strings /lib/x86_64-linux-gnu/libstdc++.so.6 2>/dev/null | grep -q 'GLIBCXX_3\.4\.32'; then
    echo "Updating libstdc++ to provide GLIBCXX_3.4.32 / CXXABI_1.3.15..."
    apt-get install -y -qq software-properties-common > /dev/null 2>&1
    add-apt-repository -y ppa:ubuntu-toolchain-r/test > /dev/null 2>&1
    apt-get update -qq
    apt-get install -y -qq libstdc++6 > /dev/null 2>&1
    ldconfig
fi

# Try cache restore
if [ -f "$LF_TAR" ]; then
    echo "Restoring LichtFeld-Studio from Drive cache..."

    # Restore source tree (needed for runtime resources, shaders, etc.)
    if [ ! -d "$LF_DIR" ]; then
        echo "Cloning LichtFeld-Studio source..."
        git clone --recursive https://github.com/MrNeRF/LichtFeld-Studio.git "$LF_DIR"
    fi

    # Restore the build dir (compiled binaries + objects + vcpkg_installed/)
    rm -rf "$LF_DIR/build"
    tar xzf "$LF_TAR" -C "$LF_DIR"

    if [ -d "$LF_DIR/build" ] && find "$LF_DIR/build" -maxdepth 3 \( -type f -executable -o -name '*.so' \) -size +0 | head -1 | grep -q .; then
        echo "LichtFeld-Studio restored from cache."
        exit 0
    fi
    echo "Cached LichtFeld build looks incomplete -- rebuilding."
    rm -f "$LF_TAR"
fi

# Build path
echo "Installing build tools..."
apt-get install -y -qq build-essential ninja-build nasm autoconf autoconf-archive automake libtool curl zip unzip tar pkg-config > /dev/null 2>&1

if ! gcc-14 --version &>/dev/null; then
    echo "Installing GCC 14..."
    apt-get install -y -qq software-properties-common > /dev/null 2>&1
    add-apt-repository -y ppa:ubuntu-toolchain-r/test > /dev/null 2>&1
    apt-get update -qq
    apt-get install -y -qq gcc-14 g++-14 > /dev/null 2>&1
    update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-14 60
    update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-14 60
fi
echo "GCC: $(gcc --version | head -1)"

CMAKE_VER="3.30.5"
if ! cmake --version 2>/dev/null | grep -q "3.3[0-9]"; then
    echo "Installing CMake ${CMAKE_VER}..."
    curl -sL "https://github.com/Kitware/CMake/releases/download/v${CMAKE_VER}/cmake-${CMAKE_VER}-linux-x86_64.tar.gz" \
        | tar xz -C /usr/local --strip-components=1
fi
echo "CMake: $(cmake --version | head -1)"

if [ ! -d "$VCPKG_ROOT" ]; then
    echo "Setting up vcpkg..."
    git clone https://github.com/microsoft/vcpkg.git "$VCPKG_ROOT"
    "$VCPKG_ROOT/bootstrap-vcpkg.sh" -disableMetrics
elif git -C "$VCPKG_ROOT" rev-parse --is-shallow-repository 2>/dev/null | grep -q true; then
    echo "Fetching full vcpkg history..."
    git -C "$VCPKG_ROOT" fetch --unshallow
fi
export VCPKG_ROOT

if [ ! -d "$LF_DIR" ]; then
    echo "Cloning LichtFeld-Studio..."
    git clone --recursive https://github.com/MrNeRF/LichtFeld-Studio.git "$LF_DIR"
fi

cd "$LF_DIR"
echo "Configuring build..."
cmake -B build -DCMAKE_BUILD_TYPE=Release -G Ninja \
    -DCMAKE_TOOLCHAIN_FILE="$VCPKG_ROOT/scripts/buildsystems/vcpkg.cmake"

echo "Compiling LichtFeld-Studio..."
cmake --build build -- -j$(nproc)

echo "Build complete"

# Save to Drive cache.
mkdir -p "$CACHE_DIR"
echo "Caching LichtFeld build to Drive..."
tar czf "$LF_TAR" -C "$LF_DIR" build
echo "  build cache: $(du -h "$LF_TAR" | cut -f1)"

## 4. Install Python Packages

In [ ]:
!pip install -q \
    fastapi==0.115.* \
    uvicorn[standard]==0.34.* \
    python-multipart \
    opencv-python-headless \
    pillow-heif \
    rawpy \
    plyfile \
    numpy \
    aiofiles \
      boto3

print("Python packages installed")

## 5. Install Cloudflared

The Cloudflare Tunnel connector routes `api.gaussiansplat.dev` to this Colab runtime.

In [ ]:
%%bash
set -e
if ! command -v cloudflared &> /dev/null; then
    curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o /tmp/cloudflared.deb
    dpkg -i /tmp/cloudflared.deb > /dev/null 2>&1
    rm /tmp/cloudflared.deb
fi
echo "cloudflared $(cloudflared --version 2>&1 | head -1)"

## 6. Install splat-transform

[playcanvas/splat-transform](https://github.com/playcanvas/splat-transform) converts `.splat` files into self-contained HTML viewers with embedded splat data.

In [ ]:
%%bash
set -e

if ! command -v splat-transform &> /dev/null; then
    npm install -g @playcanvas/splat-transform 2>&1 | tail -3
fi
echo "splat-transform $(splat-transform --version 2>&1 | head -1)"

## 7. Pipeline Server

Creates `server.py` - FastAPI server

Endpoints:

| Route | Purpose |
|---|---|
| `GET /health` | GPU & connectivity check |
| `POST /jobs/create` | Upload images, start pipeline |
| `GET /jobs/{id}/status` | Poll job progress |
| `GET /jobs/{id}/download` | Download `.ply` |
| `GET /jobs/{id}/splat` | Download `.splat` |
| `GET /view/{id}` | Self-contained 3D viewer page |

In [ ]:
%%writefile /content/server.py
"""Gaussian Splatting Pipeline Server"""

import json
import os
import re
import shutil
import struct
import subprocess
import time
import traceback
import uuid
import zipfile
from pathlib import Path
from typing import List

import boto3
from botocore.config import Config as BotoConfig

import cv2
import numpy as np
from fastapi import BackgroundTasks, FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse

# Configuration
WORKSPACE = Path('/content/jobs')
WORKSPACE.mkdir(exist_ok=True)

# R2 Storage configuration
R2_BUCKET = 'gaussian-splats'
R2_PUBLIC_URL = 'https://splats.gaussiansplat.dev'

def _get_r2_client():
    """Create an S3-compatible client for Cloudflare R2."""
    try:
        from google.colab import userdata
        access_key = userdata.get('R2_ACCESS_KEY_ID')
        secret_key = userdata.get('R2_SECRET_ACCESS_KEY')
        endpoint = userdata.get('R2_ENDPOINT_URL')
    except Exception:
        access_key = os.environ.get('R2_ACCESS_KEY_ID', '')
        secret_key = os.environ.get('R2_SECRET_ACCESS_KEY', '')
        endpoint = os.environ.get('R2_ENDPOINT_URL', '')
    if not all([access_key, secret_key, endpoint]):
        return None
    return boto3.client(
        's3',
        endpoint_url=endpoint,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        config=BotoConfig(signature_version='s3v4'),
        region_name='auto')

_R2_CONTENT_TYPES = {
    '.html': 'text/html; charset=utf-8',
}

def upload_to_r2(local_path: str, object_key: str) -> str | None:
    """Upload a file to R2 and return the public URL, or None on failure."""
    client = _get_r2_client()
    if client is None:
        print('[R2] Skipping upload - R2 credentials not configured')
        return None
    try:
        client.upload_file(
            local_path, R2_BUCKET, object_key,
            ExtraArgs={'ContentType': _R2_CONTENT_TYPES.get(Path(object_key).suffix.lower(), 'application/octet-stream')})
        url = f'{R2_PUBLIC_URL}/{object_key}'
        size_mb = Path(local_path).stat().st_size / 1048576
        print(f'[R2] Uploaded {object_key} ({size_mb:.1f} MB) -> {url}')
        return url
    except Exception as e:
        print(f'[R2] Upload failed: {e}')
        return None

LOCAL_JOBS_DIR = Path('/content/local_jobs')
JOBS_DB = Path('/content/jobs_db.json')

GS_REPO = Path('/content/gaussian-splatting')
TRAIN_SCRIPT = GS_REPO / 'train.py'
CONVERT_SCRIPT = GS_REPO / 'convert.py'

COLMAP_CAMERA_MODEL = 'OPENCV' # For iPhone lens

# FastAPI app
app = FastAPI(title='CaptureLab Gaussian Splatting Pipeline')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'])

jobs: dict = {}

# Job persistence
def save_jobs_db():
    """Persist all jobs to disk so they survive server restarts."""
    try:
        JOBS_DB.write_text(json.dumps(jobs, default=str))
    except Exception as e:
        print(f'[DB] Failed to save jobs: {e}')

def load_jobs_db():
    """Load persisted jobs from disk on startup."""
    global jobs
    if JOBS_DB.exists():
        try:
            jobs.update(json.loads(JOBS_DB.read_text()))
            print(f'[DB] Loaded {len(jobs)} jobs from {JOBS_DB}')
        except Exception as e:
            print(f'[DB] Failed to load jobs: {e}')

# Load on import
load_jobs_db()

# Utilities
def update_job(job_id: str, **kw):
    if job_id in jobs:
        jobs[job_id].update(kw)
        save_jobs_db()

def detect_blur(path: str, threshold: float = 20.0) -> tuple:
    """Return (is_blurry, laplacian_variance)"""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True, 0.0
    var = cv2.Laplacian(img, cv2.CV_64F).var()
    return var < threshold, var

# HEIC/HEIF Conversion
def convert_heic_images(input_dir: Path):
    """Convert HEIC/HEIF to JPEG"""
    try:
        from PIL import Image
        import pillow_heif
        pillow_heif.register_heif_opener()
        for p in sorted(input_dir.iterdir()):
            if p.suffix.lower() in ('.heic', '.heif'):
                img = Image.open(p)
                jpeg_path = p.with_suffix('.jpg')
                img.save(jpeg_path, 'JPEG', quality=95)
                p.unlink()
                print(f'[HEIC] {p.name} -> {jpeg_path.name}')
    except ImportError:
        print('[HEIC] pillow-heif not available, skipping HEIC conversion')

# DNG Conversion
def convert_dng_images(input_dir: Path):
    """Convert Apple DNG to JPEG for COLMAP"""
    try:
        import rawpy
        from PIL import Image
        for p in sorted(input_dir.iterdir()):
            if p.suffix.lower() == '.dng':
                with rawpy.imread(str(p)) as raw:
                    rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=False)
                jpeg_path = p.with_suffix('.jpg')
                Image.fromarray(rgb).save(jpeg_path, 'JPEG', quality=95)
                p.unlink()
                print(f'[DNG] {p.name} -> {jpeg_path.name}')
    except ImportError:
        print('[DNG] rawpy not available, skipping DNG conversion')

# Organize directories
def flatten_directory(input_dir: Path):
    """Move images from nested subdirs into *input_dir* root."""
    exts = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.dng'}
    for root, dirs, files in os.walk(input_dir):
        for fname in files:
            src = Path(root) / fname
            if src.suffix.lower() in exts and src.parent != input_dir:
                dest = input_dir / src.name
                if dest.exists():
                    dest = input_dir / f'{src.stem}_{uuid.uuid4().hex[:4]}{src.suffix}'
                shutil.move(str(src), str(dest))
    # Remove empty subdirs
    for root, dirs, _ in os.walk(input_dir, topdown=False):
        for d in dirs:
            try:
                os.rmdir(os.path.join(root, d))
            except OSError:
                pass

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.m4v', '.avi', '.mkv'}

def extract_video_frames(video_path: Path, output_dir: Path, fps: float = 2.0) -> int:
    """Extract frames from a video using ffmpeg at the given FPS."""
    output_pattern = str(output_dir / 'frame_%04d.jpg')
    result = subprocess.run(
        ['ffmpeg', '-i', str(video_path), '-vf', f'fps={fps}', '-q:v', '2', output_pattern, '-y'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'ffmpeg frame extraction failed: {result.stderr[-500:]}')
    return len(list(output_dir.glob('frame_*.jpg')))

def is_real_image_member(name: str) -> bool:
    """Check if a zip member is a real image (not a macOS resource fork)."""
    low = name.lower()
    if not low.endswith(('.jpg', '.jpeg', '.png', '.heic', '.heif', '.dng')):
        return False
    # Skip macOS resource forks (__MACOSX/._filename)
    if '__macosx' in low or '/.' in name or name.startswith('.'):
        return False
    return True

# PLY to Splat Conversion
SH_C0 = 0.28209479177387814

def ply_to_splat(ply_path: str, splat_path: str):
    """Convert 3DGS .ply to .splat"""
    from plyfile import PlyData

    pd = PlyData.read(ply_path)
    v = pd['vertex']
    n = len(v)

    # Positions
    xyz = np.column_stack([v['x'], v['y'], v['z']]).astype(np.float32)

    # Colour from 0th-order SH -> RGB [0-255]
    f_dc = np.column_stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']])
    rgb = np.clip((0.5 + SH_C0 * f_dc) * 255, 0, 255).astype(np.uint8)

    # Opacity: sigmoid -> [0-255]
    opacity_raw = np.array(v['opacity'], dtype=np.float32)
    opacity = (1.0 / (1.0 + np.exp(-opacity_raw)) * 255).astype(np.uint8)

    # Scales: stored as log-scale in .ply
    scales = np.exp(
        np.column_stack([v['scale_0'], v['scale_1'], v['scale_2']])
    ).astype(np.float32)

    # Rotation quaternion -> normalised -> [0-255]
    rot = np.column_stack(
        [v['rot_0'], v['rot_1'], v['rot_2'], v['rot_3']]
    ).astype(np.float32)
    norms = np.linalg.norm(rot, axis=1, keepdims=True)
    rot = rot / (norms + 1e-8)
    rot_u8 = np.clip(rot * 128 + 128, 0, 255).astype(np.uint8)

    # Sort by scale (volume) descending for progressive rendering
    order = np.argsort(-np.prod(scales, axis=1))

    with open(splat_path, 'wb') as f:
        for i in order:
            f.write(struct.pack('<3f', *xyz[i]))
            f.write(struct.pack('<3f', *scales[i]))
            f.write(struct.pack('4B', rgb[i, 0], rgb[i, 1], rgb[i, 2], opacity[i]))
            f.write(struct.pack('4B', *rot_u8[i]))

    size_mb = os.path.getsize(splat_path) / 1048576
    print(f'[Export] {n:,} gaussians -> {splat_path} ({size_mb:.1f} MB)')

# Splat to HTML via splat-transform
def splat_to_html(splat_path: str, html_path: str) -> bool:
    """Convert .splat to self-contained HTML viewer using splat-transform CLI."""
    try:
        r = subprocess.run(
            ['splat-transform', '-g', '0', '-w', splat_path, html_path],
            capture_output=True, text=True)
        if r.returncode == 0 and Path(html_path).exists():
            size_mb = Path(html_path).stat().st_size / 1048576
            print(f'[HTML] splat-transform -> {html_path} ({size_mb:.1f} MB)')
            return True
        print(f'[HTML] splat-transform failed: {(r.stderr or r.stdout)[-300:]}')
    except FileNotFoundError:
        print('[HTML] splat-transform not found, skipping HTML export')
    except Exception as e:
        print(f'[HTML] splat-transform error: {e}')
    return False

# COLMAP helper function
def run_colmap_gpu_match(db_path, env):
    """Run GPU exhaustive matching with CUDA"""
    gpu_cmd = [
        'colmap', 'exhaustive_matcher',
        '--database_path', str(db_path),
        '--SiftMatching.use_gpu', '1',
        '--ExhaustiveMatching.block_size', '50']
    r = subprocess.run(gpu_cmd, capture_output=True, text=True, env=env)
    if r.returncode == 0:
        print('[Match] GPU exhaustive matching succeeded')
        return True

    return (r.stderr or r.stdout or 'unknown error')[-500:]

# Pipeline function
def run_pipeline(job_id: str, job_dir: Path, blur_threshold: float):
    """
    Full pipeline: preprocess -> COLMAP SfM -> 3DGS training -> export .splat -> export HTML
    Runs in a background thread so the API stays responsive
    """
    try:
        input_dir = job_dir / 'input'
        output_dir = job_dir / 'output'

        # 0. Extract video frames if needed
        videos_dir = job_dir / 'videos'
        if videos_dir.exists():
            video_files = [v for v in sorted(videos_dir.iterdir()) if v.suffix.lower() in VIDEO_EXTENSIONS]
            if video_files:
                update_job(job_id, status='extracting', progress=5,
                           message=f'Extracting frames from {len(video_files)} video(s)â€¦')
                for vf in video_files:
                    try:
                        n = extract_video_frames(vf, input_dir)
                        print(f'[Video] Extracted {n} frames from {vf.name}')
                    except Exception as ve:
                        update_job(job_id, status='failed', progress=0,
                                   message=f'Frame extraction failed: {ve}')
                        return
                shutil.rmtree(str(videos_dir), ignore_errors=True)

        # 1. Preprocess
        update_job(job_id, status='preprocessing', progress=10,
                   message='Preprocessing images...')

        flatten_directory(input_dir)
        convert_heic_images(input_dir)
        convert_dng_images(input_dir)

        images = sorted([
            f for f in input_dir.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        ])

        # Blur detection to remove soft frames
        kept = []
        for img in images:
            blurry, var = detect_blur(str(img), blur_threshold)
            if not blurry:
                kept.append(img)
            else:
                img.unlink()
                print(f'[Blur] Removed {img.name} (var={var:.1f})')

        if len(kept) < 3:
            update_job(job_id, status='failed', progress=0,
                       message=(
                           f'Only {len(kept)} sharp images remain (need >= 3). '
                           f'Total uploaded: {len(images)}. '
                           f'Try a lower blur_threshold.'
                       ))
            return

        update_job(job_id, status='preprocessing', progress=15,
                   message=f'Kept {len(kept)}/{len(images)} images')

        # 2. COLMAP SfM
        update_job(job_id, status='sfm', progress=20,
                   message='Running COLMAP feature extraction...')

        distorted_dir = job_dir / 'distorted'
        distorted_dir.mkdir(parents=True, exist_ok=True)
        sparse_distorted = distorted_dir / 'sparse'
        sparse_distorted.mkdir(parents=True, exist_ok=True)
        db_path = distorted_dir / 'database.db'

        colmap_env = {**os.environ, 'QT_QPA_PLATFORM': 'offscreen'}

        def run_colmap(step_name, cmd):
            r = subprocess.run(cmd, capture_output=True, text=True, env=colmap_env)
            if r.returncode != 0:
                err = (r.stderr or r.stdout or 'unknown error')[-500:]
                update_job(job_id, status='failed', progress=0,
                           message=f'{step_name} failed: {err}')
                return False
            return True

        # COLMAP Step 1: Feature extraction
        if not run_colmap('Feature extraction', [
            'colmap', 'feature_extractor',
            '--database_path', str(db_path),
            '--image_path', str(input_dir),
            '--ImageReader.single_camera', '1',
            '--ImageReader.camera_model', COLMAP_CAMERA_MODEL,
            '--SiftExtraction.use_gpu', '1']):
            return

        update_job(job_id, status='sfm', progress=25,
                   message='Feature extraction complete, matching...')

        # COLMAP Step 2: Exhaustive matching
        result = run_colmap_gpu_match(
            db_path, colmap_env)
        if result is not True:
            update_job(job_id, status='failed', progress=0,
                       message=f'Feature matching failed: {result}')
            return

        update_job(job_id, status='sfm', progress=30,
                   message='Matching complete, reconstructing...')

        # COLMAP Step 3: Sparse reconstruction
        if not run_colmap('Sparse reconstruction', [
            'colmap', 'mapper',
            '--database_path', str(db_path),
            '--image_path', str(input_dir),
            '--output_path', str(sparse_distorted),
            '--Mapper.ba_global_function_tolerance', '0.000001']):
            return

        if not (sparse_distorted / '0').exists():
            update_job(job_id, status='failed', progress=0,
                       message='COLMAP mapper produced no model, images may lack overlap')
            return

        update_job(job_id, status='sfm', progress=35,
                   message='Reconstruction complete, undistorting...')

        # COLMAP Step 4: Image undistortion
        if not run_colmap('Image undistortion', [
            'colmap', 'image_undistorter',
            '--image_path', str(input_dir),
            '--input_path', str(sparse_distorted / '0'),
            '--output_path', str(job_dir),
            '--output_type', 'COLMAP']):
            return

        # Move sparse model files into sparse/0/
        final_sparse = job_dir / 'sparse'
        final_sparse_0 = final_sparse / '0'
        final_sparse_0.mkdir(parents=True, exist_ok=True)
        for f in final_sparse.iterdir():
            if f.is_file():
                shutil.move(str(f), str(final_sparse_0 / f.name))

        update_job(job_id, status='sfm', progress=40,
                   message='Camera poses estimated')

        # 3. Train 3DGS
        update_job(job_id, status='training', progress=45,
                   message='Training Gaussian Splatting...')

        # The path to the binary we compiled in Step 1
        lf_bin = '/content/LichtFeld-Studio/build/LichtFeld-Studio'

        train_cmd = [
            lf_bin,
            '-d', str(job_dir),
            '-o', str(output_dir),
            '--headless',
            '--train',
            '-i', '7000']

        proc = subprocess.Popen(
            train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True)

        # Keep the UI alive while LichtFeld trains
        for line in iter(proc.stdout.readline, ''):
            print(f"[LichtFeld] {line.strip()}")
            update_job(job_id, status='training', progress=60,
                       message='Training in progress...')

        proc.wait()

        if proc.returncode != 0:
            update_job(job_id, status='failed', progress=0,
                       message='3DGS training failed, check logs.')
            return

        update_job(job_id, status='training', progress=90,
                   message='Training complete')
        # 4. Export .splat
        update_job(job_id, status='exporting', progress=92,
                   message='Converting to web format...')

        splat_path = output_dir / f'{job_id}.splat'
        ply_path = None

        output_files = list(output_dir.glob("*.ply")) + list(output_dir.glob("*.splat"))

        if not output_files:
            update_job(job_id, status='failed', progress=0,
                       message='No output 3D file found')
            return

        primary_output = output_files[0]

        # If training output a .splat natively, copy it. If it output a .ply, convert it.
        if primary_output.suffix.lower() == '.splat':
            shutil.copy(str(primary_output), str(splat_path))
        else:
            ply_to_splat(str(primary_output), str(splat_path))
            ply_path = primary_output

        # 4. Export .splat
        update_job(job_id, status='exporting', progress=92,
                   message='Converting to web format...')

        # 5. Generate HTML viewer via splat-transform
        update_job(job_id, status='exporting', progress=95,
                   message='Generating HTML viewer...')

        html_path = output_dir / f'{job_id}.html'
        html_ok = splat_to_html(str(splat_path), str(html_path))
        if not html_ok:
            print(f'[Pipeline] HTML generation failed for {job_id}; viewer will fall '
                  f'back to /view/{job_id} (served from disk on this Colab runtime)')

        # 6. Upload the HTML viewer to R2 for persistent access.
        # Only the HTML is uploaded -- the .splat and .ply stay on the local
        # Colab runtime (served via /view/{job_id} if needed).
        html_r2_url = None
        if html_ok:
            html_r2_url = upload_to_r2(str(html_path), f'{job_id}.html')

        update_job(
            job_id,
            status='done',
            progress=100,
            message='Gaussian Splat ready!',
            ply_path=str(ply_path),
            splat_path=str(splat_path),
            html_path=str(html_path) if html_ok else '',
            viewer_url=html_r2_url or f'/view/{job_id}',
            completed_at=time.time())

    except Exception as e:
        tb = traceback.format_exc()[-500:]
        update_job(job_id, status='failed', progress=0,
                   message=f'Pipeline error: {e}\n{tb}')

# API Routes
@app.get('/health')
async def health():
    gpu = 'unknown'
    mem_gb = 0.0
    try:
        import torch
        if torch.cuda.is_available():
            gpu = torch.cuda.get_device_name(0)
            mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        else:
            gpu = 'none (CUDA unavailable)'
    except Exception as e:
        gpu = f'error: {e}'
    return {
        'status': 'ok',
        'gpu': gpu,
        'gpu_memory': f'{mem_gb:.1f}GB',
        'active_jobs': len([
            j for j in jobs.values()
            if j['status'] not in ('done', 'failed')
        ]),
        'detail': 'connected',
    }

@app.post('/jobs/create')
async def create_job(
    background_tasks: BackgroundTasks,
    files: List[UploadFile] = File(...),
    input_type: str = Form('images'),
    blur_threshold: float = Form(80.0),
    use_masks: bool = Form(True),
    session_id: str = Form('')):
    job_id = uuid.uuid4().hex[:8]
    job_dir = WORKSPACE / job_id
    input_dir = job_dir / 'input'
    input_dir.mkdir(parents=True)

    total_bytes = 0
    file_count = 0
    has_video = False

    for f in files:
        content = await f.read()
        total_bytes += len(content)

        if f.filename and f.filename.endswith('.zip'):
            # Extract zip archive
            zp = job_dir / f.filename
            zp.write_bytes(content)
            with zipfile.ZipFile(zp, 'r') as z:
                for member in z.namelist():
                    if is_real_image_member(member):
                        z.extract(member, str(input_dir))
                        file_count += 1
            zp.unlink()
        else:
            name = f.filename or f'image_{file_count}.jpg'
            ext = Path(name).suffix.lower()
            if ext in VIDEO_EXTENSIONS:
                videos_dir = job_dir / 'videos'
                videos_dir.mkdir(exist_ok=True)
                video_path = videos_dir / name
                video_path.write_bytes(content)
                has_video = True
                file_count += 1  # real frame count determined during extraction
                print(f'[Video] Saved {name} for background extraction')
            else:
                (input_dir / name).write_bytes(content)
                file_count += 1

    jobs[job_id] = {
        'job_id': job_id,
        'status': 'queued',
        'progress': 0,
        'message': 'Job queued',
        'created_at': time.time(),
    }
    save_jobs_db()

    effective_blur = min(blur_threshold, 5.0) if has_video else blur_threshold
    background_tasks.add_task(run_pipeline, job_id, job_dir, effective_blur)

    return {
        'job_id': job_id,
        'status': 'queued',
        'files_received': file_count,
        'total_mb': round(total_bytes / 1048576, 2),
        'poll_url': f'/jobs/{job_id}/status',
        'download_url': f'/jobs/{job_id}/download',
    }

@app.get('/jobs/{job_id}/status')
async def job_status(job_id: str):
    if job_id not in jobs:
        return JSONResponse(status_code=404, content={'error': 'not found'})
    return jobs[job_id]

@app.get('/jobs/{job_id}/download')
async def download_ply(job_id: str):
    if job_id not in jobs:
        return JSONResponse(status_code=404, content={'error': 'not found'})
    pp = jobs[job_id].get('ply_path')
    if not pp or not Path(pp).exists():
        return JSONResponse(status_code=404, content={'error': 'not ready'})
    return FileResponse(
        pp, filename=f'{job_id}.ply',
        media_type='application/octet-stream')

@app.get('/jobs/{job_id}/splat')
async def download_splat(job_id: str):
    if job_id not in jobs:
        return JSONResponse(status_code=404, content={'error': 'not found'})
    sp = jobs[job_id].get('splat_path')
    if not sp or not Path(sp).exists():
        return JSONResponse(status_code=404, content={'error': 'not ready'})
    return FileResponse(
        sp, filename=f'{job_id}.splat',
        media_type='application/octet-stream')

@app.get('/view/{job_id}')
async def view_splat(job_id: str):
    if job_id not in jobs:
        return HTMLResponse('<h1>Job not found</h1>', status_code=404)
    j = jobs[job_id]
    if j['status'] != 'done':
        msg = f'Job {job_id} is still processing (status: {j["status"]})'
        return HTMLResponse(f'<h1>{msg}</h1>')

    hp = j.get('html_path')
    if hp and Path(hp).exists():
        return FileResponse(hp, media_type='text/html')
    return HTMLResponse('<h1>Viewer not available, splat-transform output missing</h1>', status_code=503)

## 8. Launch Server and Cloudflare Tunnel

Starts uvicorn on port 8000, then connects the **Gaussian-Splat-API** tunnel. Be sure to add your tunnel token below to Colab Secrets.

Make sure the tunnel's public hostname is configured as:
- **Domain:** `api.gaussiansplat.dev`
- **Service:** `http://localhost:8000`

In [ ]:
import subprocess, time, os, signal

# Tunnel token + R2 credentials -- read in the Colab kernel and forward to
# the uvicorn subprocess via environment variables. `google.colab.userdata`
# is only available inside the kernel process, not in spawned subprocesses,
# so the server can't read these secrets directly.
secrets = {}
try:
    from google.colab import userdata
    TUNNEL_TOKEN = userdata.get("CLOUDFLARE_TUNNEL_TOKEN")
    print("Tunnel token loaded from Colab Secrets")
    for key in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ENDPOINT_URL"):
        try:
            val = userdata.get(key)
            if val:
                secrets[key] = val
        except Exception as e:
            print(f"  {key}: not set ({e})")
    if all(k in secrets for k in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ENDPOINT_URL")):
        print("R2 credentials loaded from Colab Secrets")
    else:
        print("R2 credentials missing -- HTML viewer will not be uploaded to R2")
except Exception:
    import getpass
    TUNNEL_TOKEN = getpass.getpass("Paste Cloudflare Tunnel token: ")

assert TUNNEL_TOKEN and len(TUNNEL_TOKEN) > 20, "Token looks invalid"

# Kill any existing server / tunnel processes
!pkill -f "uvicorn server:app" 2>/dev/null || true
!pkill -f "cloudflared tunnel" 2>/dev/null || true
time.sleep(2)

# Start FastAPI server
os.makedirs("/content/logs", exist_ok=True)

server_env = {**os.environ, **secrets}
server_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000",
     "--timeout-keep-alive", "120"],
    stdout=open("/content/logs/server.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/content",
    env=server_env,
)
print(f"Server started (PID {server_proc.pid})")

# Wait for server to be ready
import urllib.request
for i in range(15):
    time.sleep(2)
    try:
        r = urllib.request.urlopen("http://localhost:8000/health")
        if r.status == 200:
            import json
            data = json.loads(r.read())
            print(f"Server is healthy - GPU: {data.get('gpu', '?')}")
            break
    except Exception:
        if i > 0 and i % 3 == 0:
            print(f"   (still waiting... attempt {i+1}/15)")
else:
    print("Server may not be ready yet - check logs:")
    !tail -10 /content/logs/server.log

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "run", "--token", TUNNEL_TOKEN],
    stdout=open("/content/logs/tunnel.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Tunnel started (PID {tunnel_proc.pid}) at: https://api.gaussiansplat.dev")
time.sleep(5)

---
## 9. Local Pipeline (Upload Zip Manually)

If you want to process a zip file of images/videos directly in Colab **without** the app,
upload the zip to `/content/` and run the cell below.

In [ ]:
import glob, json, os, shutil, subprocess, time, uuid, zipfile, struct
from pathlib import Path
import cv2, numpy as np

os.environ["QT_QPA_PLATFORM"] = "offscreen"

ZIP_PATH = None
JOBS_DB = Path("/content/jobs_db.json")

# Auto-detect zip files
if ZIP_PATH is None:
    zips = sorted(glob.glob("/content/*.zip"))
    if zips:
        ZIP_PATH = zips[0]
        print(f"Found zip: {ZIP_PATH}")
    else:
        raise FileNotFoundError(
            "No .zip found in /content/. Upload a zip of images/videos first."
        )

# Setup directories
job_id = uuid.uuid4().hex[:8]
job_dir = Path(f"/content/local_jobs/{job_id}")
input_dir = job_dir / "input"
output_dir = job_dir / "output"
input_dir.mkdir(parents=True)

print(f"Job ID: {job_id}")
print(f"Job dir: {job_dir}")

# Extract zip: images extracted directly, videos extracted then frame-stripped with ffmpeg
VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".mkv"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".dng"}

def extract_video_frames_local(video_path, output_dir, fps=2.0):
    """Extract frames from a video file using ffmpeg."""
    pattern = str(output_dir / "frame_%04d.jpg")
    r = subprocess.run(
        ["ffmpeg", "-i", str(video_path), "-vf", f"fps={fps}", "-q:v", "2", pattern, "-y"],
        capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {r.stderr[-500:]}")
    return len(list(output_dir.glob("frame_*.jpg")))

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_members = [
        m for m in z.namelist()
        if "__macosx" not in m.lower() and "/." not in m and not m.startswith(".")
    ]
    image_members = [m for m in all_members if Path(m).suffix.lower() in IMAGE_EXTS]
    video_members = [m for m in all_members if Path(m).suffix.lower() in VIDEO_EXTS]
    print(f"Extracting {len(image_members)} images, {len(video_members)} videos...")
    for m in image_members:
        z.extract(m, str(input_dir))
    for m in video_members:
        z.extract(m, str(job_dir / "videos"))

# Extract frames from any videos
videos_dir = job_dir / "videos"
if videos_dir.exists():
    for vp in sorted(videos_dir.iterdir()):
        if vp.suffix.lower() in VIDEO_EXTS:
            try:
                n = extract_video_frames_local(vp, input_dir)
                print(f"  {vp.name}: extracted {n} frames")
            except Exception as e:
                print(f"  {vp.name}: frame extraction failed â€” {e}")

# Flatten nested dirs
exts = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".dng"}
for root, dirs, files in os.walk(input_dir):
    for fname in files:
        src = Path(root) / fname
        if src.suffix.lower() in exts and src.parent != input_dir:
            dest = input_dir / src.name
            if dest.exists():
                dest = input_dir / f"{src.stem}_{uuid.uuid4().hex[:4]}{src.suffix}"
            shutil.move(str(src), str(dest))

# Convert HEIC/DNG to JPEG
try:
    from PIL import Image
    import pillow_heif
    pillow_heif.register_heif_opener()
    for p in sorted(input_dir.iterdir()):
        if p.suffix.lower() in (".heic", ".heif"):
            img = Image.open(p)
            img.save(p.with_suffix(".jpg"), "JPEG", quality=95)
            p.unlink()
            print(f"  HEIC -> {p.with_suffix('.jpg').name}")
except ImportError:
    pass

try:
    import rawpy
    from PIL import Image
    for p in sorted(input_dir.iterdir()):
        if p.suffix.lower() == ".dng":
            with rawpy.imread(str(p)) as raw:
                rgb = raw.postprocess(use_camera_wb=True)
            Image.fromarray(rgb).save(p.with_suffix(".jpg"), "JPEG", quality=95)
            p.unlink()
except ImportError:
    pass

# Blur filter
BLUR_THRESHOLD = 15.0
VIDEO_INPUT = any(
    Path(m).suffix.lower() in VIDEO_EXTS
    for m in zipfile.ZipFile(ZIP_PATH).namelist()
    if "__macosx" not in m.lower()
)
if VIDEO_INPUT:
    BLUR_THRESHOLD = 5.0
    print(f"Video input detected â€” using blur threshold {BLUR_THRESHOLD}")
images = sorted([
    f for f in input_dir.iterdir()
    if f.suffix.lower() in (".jpg", ".jpeg", ".png")
])
kept = []
for img_path in images:
    gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        continue
    var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if var >= BLUR_THRESHOLD:
        kept.append(img_path)
    else:
        img_path.unlink()
        print(f"  Blur removed: {img_path.name} (var={var:.1f})")

print(f"\nKept {len(kept)}/{len(images)} sharp images")
assert len(kept) >= 3, f"Need at least 3 sharp images, got {len(kept)}"

# Helper to run a command with error checking
def run_step(name, cmd):
    """Run a subprocess, raise with full output on failure."""
    print(f"  {name}...")
    env = {**os.environ, "QT_QPA_PLATFORM": "offscreen"}
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(f" {name} FAILED (exit code {r.returncode})")
        if r.stdout:
            print("-- stdout (last 2000 chars) --")
            print(r.stdout[-2000:])
        if r.stderr:
            print("-- stderr (last 2000 chars) --")
            print(r.stderr[-2000:])
        raise RuntimeError(f"{name} failed -- see output above")
    return r

# COLMAP
print("\nRunning COLMAP...")

distorted_dir = job_dir / "distorted"
distorted_dir.mkdir(parents=True, exist_ok=True)
sparse_distorted = distorted_dir / "sparse"
sparse_distorted.mkdir(parents=True, exist_ok=True)
db_path = distorted_dir / "database.db"
colmap_env = {**os.environ, "QT_QPA_PLATFORM": "offscreen"}

# Step 1: Feature extraction
run_step("Step 1/4: Feature extraction", [
    "colmap", "feature_extractor",
    "--database_path", str(db_path),
    "--image_path", str(input_dir),
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "OPENCV",
    "--SiftExtraction.use_gpu", "1"])
print("  Features extracted")

# Step 2: Exhaustive matching
run_step("Step 2/4: Exhaustive matching", [
    "colmap", "exhaustive_matcher",
    "--database_path", str(db_path),
    "--SiftMatching.use_gpu", "1",
    "--ExhaustiveMatching.block_size", "50"])
print("  Features matched")

# Step 3: Sparse reconstruction (mapper)
run_step("Step 3/4: Sparse reconstruction (mapper)", [
    "colmap", "mapper",
    "--database_path", str(db_path),
    "--image_path", str(input_dir),
    "--output_path", str(sparse_distorted),
    "--Mapper.ba_global_function_tolerance", "0.000001"])

# Verify sparse model was created
if not (sparse_distorted / "0").exists():
    print("Mapper completed but no reconstruction was produced")
    raise RuntimeError("COLMAP mapper produced no model. Images may lack sufficient overlap.")
print("  Sparse model reconstructed")

# Step 4: Image undistortion
run_step("Step 4/4: Image undistortion", [
    "colmap", "image_undistorter",
    "--image_path", str(input_dir),
    "--input_path", str(sparse_distorted / "0"),
    "--output_path", str(job_dir),
    "--output_type", "COLMAP"])

# Move sparse model files into sparse/0/
final_sparse = job_dir / "sparse"
final_sparse_0 = final_sparse / "0"
final_sparse_0.mkdir(parents=True, exist_ok=True)
for f in final_sparse.iterdir():
    if f.is_file():
        shutil.move(str(f), str(final_sparse_0 / f.name))

print("Camera poses estimated - COLMAP complete")

# Verify final structure
for expected in ["cameras.bin", "images.bin", "points3D.bin"]:
    p = final_sparse_0 / expected
    if not p.exists():
        raise RuntimeError(f"Missing {expected} in {final_sparse_0}")

# Train 3D Gaussian Splatting
print(f"\nTraining 3DGS...")

lf_bin = '/content/LichtFeld-Studio/build/LichtFeld-Studio'
train_cmd = [
    lf_bin,
    '-d', str(job_dir),
    '-o', str(output_dir),
    '--headless',
    '--train',
    '-i', '7000']

train_result = subprocess.run(
    train_cmd, capture_output=True, text=True)

if train_result.returncode != 0:
    print("TRAINING FAILED!")
    print("-- stdout (last 2000 chars) --")
    print(train_result.stdout[-2000:] if train_result.stdout else "(empty)")
    print("-- stderr (last 2000 chars) --")
    print(train_result.stderr[-2000:] if train_result.stderr else "(empty)")
    raise RuntimeError("3DGS training failed -- check output above")

print("Training complete")

# Export .splat
output_files = list(output_dir.glob("*.ply")) + list(output_dir.glob("*.splat"))
assert len(output_files) > 0, "No output file found!"

primary_output = output_files[0]
splat_path = output_dir / f"{job_id}.splat"
ply_path = None

if primary_output.suffix.lower() == '.splat':
    shutil.copy(str(primary_output), str(splat_path))
    ply_path = primary_output
else:
    # Convert .ply to .splat if output is a ply
    from plyfile import PlyData
    SH_C0 = 0.28209479177387814
    pd = PlyData.read(str(primary_output))
    v = pd["vertex"]
    n = len(v)

    xyz = np.column_stack([v["x"], v["y"], v["z"]]).astype(np.float32)
    f_dc = np.column_stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]])
    rgb = np.clip((0.5 + SH_C0 * f_dc) * 255, 0, 255).astype(np.uint8)
    opacity = (1.0 / (1.0 + np.exp(-np.array(v["opacity"], dtype=np.float32))) * 255).astype(np.uint8)
    scales = np.exp(np.column_stack([v["scale_0"], v["scale_1"], v["scale_2"]])).astype(np.float32)
    rot = np.column_stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]]).astype(np.float32)
    norms = np.linalg.norm(rot, axis=1, keepdims=True)
    rot = rot / (norms + 1e-8)
    rot_u8 = np.clip(rot * 128 + 128, 0, 255).astype(np.uint8)
    order = np.argsort(-np.prod(scales, axis=1))

    with open(str(splat_path), "wb") as f:
        for i in order:
            f.write(struct.pack("<3f", *xyz[i]))
            f.write(struct.pack("<3f", *scales[i]))
            f.write(struct.pack("4B", rgb[i, 0], rgb[i, 1], rgb[i, 2], opacity[i]))
            f.write(struct.pack("4B", *rot_u8[i]))

    ply_path = primary_output
    print(f"\nConverted PLY to SPLAT -> {splat_path}")

size_mb = splat_path.stat().st_size / 1048576
print(f"Final Model Size: {size_mb:.1f} MB")

# Generate HTML viewer via splat-transform
print("\nGenerating HTML viewer via splat-transform...")
html_path = output_dir / f"{job_id}.html"
html_ok = False
try:
    html_result = subprocess.run(
        ["splat-transform", "-g", "0", "-w", str(splat_path), str(html_path)],
        capture_output=True, text=True)
    if html_result.returncode == 0 and html_path.exists():
        html_size_mb = html_path.stat().st_size / 1048576
        print(f"HTML viewer: {html_path} ({html_size_mb:.1f} MB)")
        html_ok = True
    else:
        print(f"splat-transform failed: {(html_result.stderr or html_result.stdout)[-300:]}")
except FileNotFoundError:
    print("splat-transform not found -- run cell 5b to install it")
except Exception as e:
    print(f"splat-transform error: {e}")

# Upload HTML to R2 for persistent access
r2_url = None
if html_ok:
    try:
        import boto3
        from botocore.config import Config as BotoConfig
        try:
            from google.colab import userdata
            r2_ak = userdata.get('R2_ACCESS_KEY_ID')
            r2_sk = userdata.get('R2_SECRET_ACCESS_KEY')
            r2_ep = userdata.get('R2_ENDPOINT_URL')
        except Exception:
            r2_ak = os.environ.get('R2_ACCESS_KEY_ID', '')
            r2_sk = os.environ.get('R2_SECRET_ACCESS_KEY', '')
            r2_ep = os.environ.get('R2_ENDPOINT_URL', '')
        if all([r2_ak, r2_sk, r2_ep]):
            s3 = boto3.client('s3', endpoint_url=r2_ep,
                              aws_access_key_id=r2_ak, aws_secret_access_key=r2_sk,
                              config=BotoConfig(signature_version='s3v4'), region_name='auto')
            s3.upload_file(str(html_path), 'gaussian-splats', f'{job_id}.html',
                           ExtraArgs={'ContentType': 'text/html'})
            r2_url = f'https://splats.gaussiansplat.dev/{job_id}.html'
            print(f"Uploaded to R2: {r2_url}")
        else:
            print("R2 credentials not configured, skipping upload")
    except Exception as e:
        print(f"R2 upload failed: {e}")

# Persist job to shared DB so the server can find it
job_data = {
    "job_id": job_id,
    "status": "done",
    "progress": 100,
    "message": "Local pipeline complete",
    "ply_path": str(ply_path),
    "splat_path": str(splat_path),
    "html_path": str(html_path) if html_ok else "",
    "viewer_url": r2_url or f"/view/{job_id}",
    "completed_at": time.time(),
}

# Load existing DB, merge, save
all_jobs = {}
if JOBS_DB.exists():
    try:
        all_jobs = json.loads(JOBS_DB.read_text())
    except Exception:
        pass
all_jobs[job_id] = job_data
JOBS_DB.write_text(json.dumps(all_jobs, default=str))
print(f"Job saved to {JOBS_DB}")

# Also register in-memory with the running server (if importable)
try:
    import importlib, sys
    sys.path.insert(0, "/content")
    import server as srv
    srv.jobs[job_id] = job_data
    print("Registered with running server (in-memory)")
except Exception:
    pass

if r2_url:
    print(f"\nView splat here: {r2_url}")
else:
    print(f"\nView via tunnel: https://api.gaussiansplat.dev/view/{job_id}")
    print(f"\nNote: If the server was restarted, re-run cell 7 (Launch Server)")